# PRW — DINOv3 last-two-block fine-tuning, random P-K, InfoNCE

Frozen Swin detector; train the last two DINOv3 blocks, final norm and projection. Use random P=8/K=3 identity groups, no offline mining, supervised multi-positive InfoNCE at temperature 0.1, and original-pixel crops resized to 256×128 for both training and inference. Validation remains P=4/K=2.

Set detector and official DINO weight paths before running. Start fresh; the checkpoint stores crop resolution for automatic inference reconstruction. All labeled people in sampled frames contribute, so crop count may exceed P×K. This experiment changes sampling, batch size and resolution relative to the preceding InfoNCE run.

Best checkpoints now maximize fixed validation-gallery retrieval mAP (every epoch by default). Only held-out training frames are used; frozen detector boxes are cached once and embeddings recomputed each evaluation. Final official test evaluation remains after training. Keep last two blocks and P8/K3 for this checkpoint-selection comparison. GPU peak allocated/reserved GiB are logged per training epoch.


In [ ]:
from kaggle_secrets import UserSecretsClient
import base64
import os
import subprocess
import sys
from pathlib import Path

USER = "SaadQAbbasi"
REPO = "ML4CV-Assignment"
repo_path = f"/kaggle/working/{REPO}"
repo_dir = Path(repo_path)
remote = f"https://github.com/{USER}/{REPO}.git"

# Pass credentials through the child environment, not the URL or notebook output.
token = UserSecretsClient().get_secret("GITHUB_TOKEN")
auth = base64.b64encode(f"x-access-token:{token}".encode()).decode()
git_env = os.environ.copy()
git_env.update({
    "GIT_TERMINAL_PROMPT": "0",
    "GIT_CONFIG_COUNT": "1",
    "GIT_CONFIG_KEY_0": "http.https://github.com/.extraheader",
    "GIT_CONFIG_VALUE_0": f"Authorization: Basic {auth}",
})
try:
    if not repo_dir.exists():
        command = ["git", "clone", remote, str(repo_dir)]
    elif (repo_dir / ".git").is_dir():
        command = ["git", "-C", str(repo_dir), "pull", "--ff-only", remote]
    else:
        raise RuntimeError(f"{repo_dir} exists but is not a Git checkout.")
    result = subprocess.run(command, env=git_env, capture_output=True, text=True)
    if result.returncode:
        detail = (result.stderr + result.stdout).replace(token, "[REDACTED]").replace(auth, "[REDACTED]")
        raise RuntimeError(
            "Repository setup failed. Check Kaggle Internet access, GITHUB_TOKEN "
            "secret access for this notebook, and repository permissions.\n" + detail
        )
finally:
    del token, auth, git_env

src_dir = repo_dir / "src"
required = ("model.py", "prw_dataset.py", "training.py", "prw_evaluation.py", "dino_reid.py", "dinov3_reid.py", "offline_mining.py")
missing = [name for name in required if not (src_dir / name).is_file()]
if missing:
    raise RuntimeError(f"Repository at {repo_dir} is missing source files: {missing}")
src_path = str(src_dir.resolve())
if src_path in sys.path:
    sys.path.remove(src_path)
sys.path.insert(0, src_path)
import importlib
importlib.invalidate_caches()
print(f"Project imports configured from {src_path}")


In [ ]:


from kaggle_secrets import UserSecretsClient
import wandb

secrets = UserSecretsClient()
wandb_api_key = secrets.get_secret("WAND_B_API_KEY")

wandb.login(key=wandb_api_key)




PRW_ROOT = "/kaggle/input/datasets/edoardomerli/prw-person-re-identification-in-the-wild"

data_root = PRW_ROOT



In [ ]:
import random
import torch
import wandb
from kaggle_secrets import UserSecretsClient
from torch.utils.data import DataLoader

from model import PersonSearchModel
from prw_dataset import (PRWPersonSearchDataset, PRWSubset,
                         PKIdentityBatchSampler, prw_collate_fn)
from training import make_grad_scaler, start_run, train_one_epoch, validate


SEED = 0
torch.manual_seed(SEED)
random.seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)
if device.type == 'cuda':
    print(torch.cuda.get_device_name(0))
    print(f'GPU memory: {torch.cuda.get_device_properties(0).total_memory / 2**30:.2f} GiB')

## Load and freeze the verified Swin detector
Set the checkpoint path before Run All. This does not resume detector training.

In [ ]:
from dino_reid import DinoPersonSearch, crop_epoch
from dinov3_reid import DinoV3CropReID

DETECTOR_CHECKPOINT = Path('/kaggle/input/YOUR-SWIN-CHECKPOINT/best.pt')
detector_checkpoint = torch.load(DETECTOR_CHECKPOINT, map_location='cpu', weights_only=False)
detector_config = dict(detector_checkpoint['model_config'])
if detector_config.get('backbone_name') != 'swin_t' or detector_config.get('metric_loss') != 'triplet':
    raise ValueError('Expected the verified Swin-Tiny + triplet detector checkpoint')
print('Detector epoch:', detector_checkpoint['epoch'], 'config:', detector_config)
detector_config['pretrained'] = False
detector = PersonSearchModel(**detector_config).to(device)
detector.load_state_dict(detector_checkpoint['model_state_dict'], strict=True)
detector.requires_grad_(False).eval()
DETECTOR_EPOCH = detector_checkpoint['epoch']
# Reproduce the detector experiment's split; never train on its validation frames.
SEED = detector_checkpoint['run_config']['seed']
if tuple(detector_checkpoint['run_config']['image_size']) != (256, 384):
    raise ValueError('This notebook assumes the baseline 256x384 scene size')
del detector_checkpoint
torch.manual_seed(SEED)
random.seed(SEED)


In [ ]:
IMAGE_SIZE = (256, 384)
REID_CROP_SIZE = (256, 128)
VAL_FRACTION = 0.10
P, K = 8, 3
VAL_P, VAL_K = 4, 2  # smaller fixed batches; every selected ID has a positive
NUM_WORKERS = 2

full_train = PRWPersonSearchDataset(PRW_ROOT, split='train', image_size=IMAGE_SIZE, query_crop_size=REID_CROP_SIZE)
generator = torch.Generator().manual_seed(SEED)
indices = torch.randperm(len(full_train), generator=generator).tolist()
val_count = round(len(indices) * VAL_FRACTION)
val_dataset = PRWSubset(full_train, indices[:val_count])
train_dataset = PRWSubset(full_train, indices[val_count:])
train_sampler = PKIdentityBatchSampler(train_dataset, p=P, k=K)

train_loader = DataLoader(
    train_dataset, batch_sampler=train_sampler, collate_fn=prw_collate_fn,
    num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=NUM_WORKERS > 0,
)
val_sampler = PKIdentityBatchSampler(val_dataset, p=VAL_P, k=VAL_K, seed=SEED)

val_loader = DataLoader(
    val_dataset, batch_sampler=val_sampler, collate_fn=prw_collate_fn,
    num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=NUM_WORKERS > 0,
)
print(f'train frames: {len(train_dataset)}, validation frames: {len(val_dataset)}')
print(f'PK batches per epoch: {len(train_sampler)}, nominal batch size: {P*K}')
print(f'Fixed validation PK batches: {len(val_sampler)}; eligible identities: '
      f'{len(val_sampler.eligible_identities)}')

In [ ]:
from offline_mining import MinedPKIdentityBatchSampler, mine_training_neighbors

MINING_ENABLED = False  # False: fine-tuning-only control
MINING_WARMUP_EPOCHS = 5
MINING_REFRESH_EVERY = 5
MINING_TOP_K = 20
HARD_FRACTION = .5
mining_config = dict(enabled=MINING_ENABLED, warmup_epochs=MINING_WARMUP_EPOCHS,
                     refresh_every=MINING_REFRESH_EVERY, top_k=MINING_TOP_K,
                     hard_fraction=HARD_FRACTION, aggregation='max_crop_cosine')
if MINING_REFRESH_EVERY < 1 or MINING_WARMUP_EPOCHS < 0:
    raise ValueError('Invalid mining schedule')
train_sampler = MinedPKIdentityBatchSampler(train_dataset, p=P, k=K, seed=SEED,
                                          hard_fraction=HARD_FRACTION)
train_loader = DataLoader(train_dataset, batch_sampler=train_sampler,
                         collate_fn=prw_collate_fn, num_workers=NUM_WORKERS,
                         pin_memory=True, persistent_workers=NUM_WORKERS > 0)
# Sequential traversal embeds ALL training frames, independent of sampled epochs.
# Use a separate generator so mining does not consume the training RNG stream.
mining_loader = DataLoader(train_dataset, batch_size=2, shuffle=False,
                          collate_fn=prw_collate_fn, num_workers=NUM_WORKERS,
                          pin_memory=True, persistent_workers=NUM_WORKERS > 0,
                          generator=torch.Generator().manual_seed(SEED + 1000))
print('Mining uses only', len(train_dataset), 'training frames; validation stays unchanged.')


## Fine-tune DINOv3 and train the projection
Last two blocks and final normalization use LR 1e-5; projection LR 1e-4. Swin remains frozen. Checkpoints include mining state and RNG state for resumption.


In [ ]:
import json
from validation_retrieval import build_validation_gallery, evaluate_validation_gallery
VAL_RETRIEVAL_EVERY = 1  # every epoch; increase if validation is too slow
NUM_EPOCHS = 50
VAL_MAX_BATCHES = 20
DINO_WEIGHTS = Path('/kaggle/input/YOUR-DINOV3-WEIGHTS/dinov3_vits16.pth')
TRAIN_LAST_BLOCKS = 2
DINO_LR = 1e-5
HEAD_LR = 1e-4
TRIPLET_MARGIN = .3  # unused by InfoNCE
METRIC_LOSS = "supcon"  # supervised multi-positive InfoNCE / NT-Xent
REID_TEMPERATURE = .1
RESUME_FROM = None
WANDB_CHECKPOINT_EVERY = 5
# For reproducibility, replace main with a specific upstream revision if desired.
DINO_HUB_REPO = 'facebookresearch/dinov3:main'
reid_config = dict(hub_repo=DINO_HUB_REPO, embedding_dim=256, train_last_blocks=TRAIN_LAST_BLOCKS, crop_size=REID_CROP_SIZE)
reid = DinoV3CropReID(**reid_config, weights_path=DINO_WEIGHTS).to(device)
assert any(p.requires_grad for p in reid.backbone.parameters())
model = DinoPersonSearch(detector, reid)
run_config = dict(experiment='swin_frozen_dinov3_last2_random_supcon_p8k3_256x128', seed=SEED, val_fraction=VAL_FRACTION,
                  image_size=IMAGE_SIZE, query_crop_size=REID_CROP_SIZE, P=P, K=K, epochs=NUM_EPOCHS,
                  reid_config=reid_config, detector_config=detector_config,
                  detector_checkpoint=str(DETECTOR_CHECKPOINT), detector_epoch=DETECTOR_EPOCH,
                  dino_frozen=False, dino_lr=DINO_LR, mining_config=mining_config, dino_weights=str(DINO_WEIGHTS), head_lr=HEAD_LR, triplet_margin=TRIPLET_MARGIN, metric_loss=METRIC_LOSS, reid_temperature=REID_TEMPERATURE,
                  gallery_embeddings='standalone_detected_crops', score_threshold=.5, checkpoint_selection='validation_retrieval_mAP',
                  val_retrieval_every=VAL_RETRIEVAL_EVERY)
optimizer = torch.optim.AdamW([
    {'params': [p for p in reid.backbone.parameters() if p.requires_grad], 'lr': DINO_LR},
    {'params': reid.embedding_proj.parameters(), 'lr': HEAD_LR},
], weight_decay=1e-4)
scaler = make_grad_scaler(device)
checkpoint_dir = Path(repo_path) / 'checkpoints_dinov3_random_infonce_256x128_valmap'
checkpoint_dir.mkdir(parents=True, exist_ok=True)
start_epoch, best_val = 0, float('inf')
best_retrieval_map = -1.
last_mining_epoch = None
mined_neighbors = {}
if RESUME_FROM is not None:
    state = torch.load(RESUME_FROM, map_location='cpu', weights_only=False)
    if (state['run_config'].get('experiment') != run_config['experiment'] or
        any(state['run_config'].get(k) != run_config[k] for k in ('P', 'K', 'metric_loss', 'reid_temperature'))):
        raise ValueError('Resume experiment, batch size or loss settings mismatch')
    if state['reid_config'] != reid_config or state['detector_config'] != detector_config:
        raise ValueError('Resume configuration mismatch')
    if state['run_config'].get('mining_config') != mining_config or state['run_config']['seed'] != SEED:
        raise ValueError('Resume mining settings or split seed mismatch')
    if state['run_config'].get('checkpoint_selection') != 'validation_retrieval_mAP':
        raise ValueError('Start fresh: previous checkpoint selected by batch loss')
    if state['run_config'].get('val_retrieval_every') != VAL_RETRIEVAL_EVERY:
        raise ValueError('Resume validation schedule mismatch')
    best_retrieval_map = state['best_retrieval_map']
    mined_neighbors = state.get('mined_neighbors', {})
    train_sampler.set_neighbors(mined_neighbors)
    last_mining_epoch = state.get('last_mining_epoch')
    random.setstate(state['python_rng'])
    torch.random.set_rng_state(state['torch_rng'])
    if torch.cuda.is_available() and state['cuda_rng'] is not None:
        torch.cuda.set_rng_state_all(state['cuda_rng'])
    model.detector.load_state_dict(state['detector_state_dict'])
    reid.load_state_dict(state['reid_state_dict'])
    optimizer.load_state_dict(state['optimizer_state_dict'])
    scaler.load_state_dict(state['scaler_state_dict'])
    start_epoch, best_val = state['epoch'], state['best_val']
    del state
run = start_run(run_config, name='prw-dinov3-last2-random-infonce-p8-k3-256x128', mode='online')
validation_names = [full_train.image_names[i] for i in val_dataset.indices]
validation_cache = build_validation_gallery(model, PRW_ROOT, validation_names, device, score_threshold=.5)
torch.save(validation_cache, checkpoint_dir / 'validation_gallery.pt')
print('Fixed validation gallery:', len(validation_names), 'frames,', len(validation_cache['queries']), 'queries')
for epoch in range(start_epoch, NUM_EPOCHS):
    if device.type == 'cuda':
        torch.cuda.reset_peak_memory_stats(device)
    train_sampler.set_epoch(epoch)
    # Warm-up uses random identity groups while the new projection begins learning.
    refresh = (epoch >= MINING_WARMUP_EPOCHS and
               (last_mining_epoch is None or epoch - last_mining_epoch >= MINING_REFRESH_EVERY))
    if MINING_ENABLED and refresh:
        mined_neighbors, mining_stats = mine_training_neighbors(
            reid, mining_loader, device, train_sampler.eligible_identities,
            top_k=MINING_TOP_K, crop_batch_size=64, similarity_chunk_size=256)
        train_sampler.set_neighbors(mined_neighbors)
        last_mining_epoch = epoch
        run.log({'epoch': epoch, **{f'mining/{k}': v for k,v in mining_stats.items()}})
        (checkpoint_dir / 'mining_neighbors.json').write_text(json.dumps({
            'epoch': epoch, 'settings': mining_config, 'neighbors': mined_neighbors}, indent=2))
        print(f'Refreshed training-only neighbor lists before epoch {epoch+1}', flush=True)
    crop_epoch(reid, train_loader, device, optimizer, scaler, TRIPLET_MARGIN, run, epoch,
               metric_loss=METRIC_LOSS, temperature=REID_TEMPERATURE)
    if device.type == 'cuda':
        run.log({'epoch': epoch+1, 'train/peak_allocated_gib': torch.cuda.max_memory_allocated(device)/2**30,
                 'train/peak_reserved_gib': torch.cuda.max_memory_reserved(device)/2**30})
    run.log({'epoch': epoch+1, **{f'mining/{k}': v for k,v in train_sampler.last_stats.items()}})
    metrics = crop_epoch(reid, val_loader, device, margin=TRIPLET_MARGIN,
                         run=run, epoch=epoch, max_batches=VAL_MAX_BATCHES,
                         metric_loss=METRIC_LOSS, temperature=REID_TEMPERATURE)
    improved = False
    retrieval_val = None
    if (epoch+1) % VAL_RETRIEVAL_EVERY == 0 or epoch == 0 or epoch+1 == NUM_EPOCHS:
        retrieval_val = evaluate_validation_gallery(model, PRW_ROOT, validation_cache, device)
        run.log({'epoch': epoch+1, **{f'val/{k}': v for k,v in retrieval_val.items()}})
        improved = retrieval_val['retrieval_mAP'] > best_retrieval_map
        best_retrieval_map = max(best_retrieval_map, retrieval_val['retrieval_mAP'])
        print('Validation retrieval:', retrieval_val, flush=True)
    best_val = min(best_val, metrics['metric_loss'])
    state = dict(epoch=epoch+1, best_val=best_val, best_retrieval_map=best_retrieval_map,
                 validation_retrieval=retrieval_val, run_config=run_config,
                 reid_config=reid_config, detector_config=detector_config,
                 detector_state_dict=model.detector.state_dict(), reid_state_dict=reid.state_dict(),
                 optimizer_state_dict=optimizer.state_dict(), scaler_state_dict=scaler.state_dict(),
                 mined_neighbors=mined_neighbors, last_mining_epoch=last_mining_epoch,
                 python_rng=random.getstate(), torch_rng=torch.random.get_rng_state(),
                 cuda_rng=torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None)
    torch.save(state, checkpoint_dir / 'latest.pt')
    if improved:
        torch.save(state, checkpoint_dir / 'best.pt')
    if (epoch+1) % WANDB_CHECKPOINT_EVERY == 0 or epoch+1 == NUM_EPOCHS:
        # Persist best periodically as well, so a late crash does not lose it.
        for filename, collection in [('best.pt', 'prw-dinov3-random-infonce-256x128-valmap-best-model'), ('latest.pt', 'prw-dinov3-random-infonce-256x128-valmap-training-checkpoint')]:
            artifact = wandb.Artifact(collection, type='model', metadata={'upload_epoch': epoch+1})
            artifact.add_file(str(checkpoint_dir / filename))
            run.log_artifact(artifact)
    print(f'Epoch {epoch+1}: val={metrics["metric_loss"]:.4f}, best retrieval mAP={best_retrieval_map:.4f}', flush=True)


## Final PRW test evaluation
Automatic mode loads this experiment's local best.pt and logs to the active run. For manual evaluation run setup/imports only, uncomment the block below, then run the shared evaluation cell. Supply a downloaded DINO experiment checkpoint (not the original Swin checkpoint) and its training run ID. No optimization is performed during evaluation.


In [ ]:
# Optional manual mode: uncomment and edit.
# DINO_EVAL_CHECKPOINT = Path('/kaggle/input/YOUR-DINO-CHECKPOINT/best.pt')
# DINO_EVAL_RUN_ID = 'REPLACE_WITH_DINO_RUN_ID'
# if wandb.run is not None:
#     wandb.finish()
# run = wandb.init(entity='models-alma-mater-studio', project='person-search',
#                  id=DINO_EVAL_RUN_ID, resume='must')


In [ ]:
from dino_reid import DinoPersonSearch
from dinov3_reid import DinoV3CropReID
import importlib
import prw_evaluation
importlib.reload(prw_evaluation)
from prw_evaluation import evaluate_prw, make_wandb_retrieval_table
if wandb.run is None:
    raise RuntimeError('Run training or configure manual evaluation first')
path = globals().pop('DINO_EVAL_CHECKPOINT', None)
ckpt_path = Path(path) if path is not None else checkpoint_dir / 'best.pt'
checkpoint = torch.load(ckpt_path, map_location='cpu', weights_only=False)
if checkpoint['run_config']['experiment'] != 'swin_frozen_dinov3_last2_random_supcon_p8k3_256x128':
    raise ValueError('Not a DINO experiment checkpoint')
# Reconstruct both networks from checkpoint configuration and exact saved weights.
for name in ('model', 'detector', 'reid', 'optimizer'):
    globals().pop(name, None)
if torch.cuda.is_available():
    torch.cuda.empty_cache()
detector = PersonSearchModel(**checkpoint['detector_config'])
reid = DinoV3CropReID(**checkpoint['reid_config'], pretrained=False)
model = DinoPersonSearch(detector, reid).to(device)
model.detector.load_state_dict(checkpoint['detector_state_dict'], strict=True)
model.reid_head.load_state_dict(checkpoint['reid_state_dict'], strict=True)
model.eval()
provenance = {'test/checkpoint_path': str(ckpt_path), 'test/checkpoint_epoch': checkpoint['epoch'],
              'test/backbone': 'finetuned dinov3_vits16 + frozen swin_t detector', 'test/metric_loss': checkpoint['run_config']['metric_loss'],
              'test/reid_temperature': checkpoint['run_config']['reid_temperature'],
              'test/detector_epoch': checkpoint['run_config']['detector_epoch'],
              'test/score_threshold': .5}
del checkpoint
print('Evaluating', provenance)
run = wandb.run
# An explicit evaluation axis gives test metrics a chart point on resumed runs too.
evaluation_index = int(run.summary.get("test/evaluation_index", 0)) + 1
run.define_metric("test/evaluation_index")
run.define_metric("test/*", step_metric="test/evaluation_index")
retrieval_metrics = evaluate_prw(
    model, PRW_ROOT, device, batch_size=2, score_threshold=0.5,
    iou_threshold=0.5, return_records=True, table_queries=100, table_top_k=5,
    query_batch_size=64, ranking_batch_size=8, gallery_crop_batch_size=32,
)
table_records = retrieval_metrics.pop("retrieval_records")
test_summary = {
    "test/mAP": retrieval_metrics["retrieval_mAP"],
    "test/top1": retrieval_metrics["retrieval_top1"],
    "test/evaluated_queries": retrieval_metrics["retrieval_queries"],
    "test/gallery_frames": retrieval_metrics["retrieval_gallery_frames"],
    "test/detections": retrieval_metrics["retrieval_detections"],
    "test/evaluation_index": evaluation_index,
}
# Persist scalar results BEFORE creating/uploading optional media.
import json
results_dir = Path("/kaggle/working/evaluation_results")
results_dir.mkdir(parents=True, exist_ok=True)
results_path = results_dir / f"{run.id}-evaluation-{evaluation_index}.json"
results_path.write_text(json.dumps({**test_summary, **provenance}, indent=2))
for metric in ("test/mAP", "test/top1"):
    run.define_metric(metric, step_metric="test/evaluation_index", summary="copy")
for key, value in {**test_summary, **provenance}.items():
    run.summary[key] = value
run.log({**test_summary, **provenance}, commit=True)
# Explicit table and bar chart: a single test result is easy to miss on line plots.
final_scores = wandb.Table(
    columns=["metric", "value", "percent", "checkpoint_epoch", "backbone", "loss"],
    data=[[name, test_summary[key], 100 * test_summary[key],
           provenance["test/checkpoint_epoch"], provenance["test/backbone"], provenance["test/metric_loss"]]
          for name, key in (("mAP", "test/mAP"), ("Top-1", "test/top1"))],
)
run.log({
    "test/evaluation_index": evaluation_index,
    "test/final_scores": final_scores,
    "test/final_scores_chart": wandb.plot.bar(
        final_scores, "metric", "percent", title="Final PRW test scores (%)"),
})
from IPython.display import display
display(final_scores.get_dataframe())
print(f"FINAL PRW TEST: mAP={test_summary['test/mAP']:.4f}, top-1={test_summary['test/top1']:.4f}")
print(f"Scalar results logged to {run.url}; local backup: {results_path}")

try:
    retrieval_table = make_wandb_retrieval_table(table_records, PRW_ROOT)
    # Log explicit image comparisons as well as the interactive table.
    # Convert rendered figures to raster images so W&B displays an image gallery.
    import io
    import matplotlib.pyplot as plt
    from PIL import Image
    from prw_evaluation import plot_retrieval_query
    
    PREVIEW_QUERIES = 10  # table still includes up to 100 queries
    comparison_images = []
    for query_index in range(min(PREVIEW_QUERIES, len(table_records))):
        fig = plot_retrieval_query(table_records, PRW_ROOT, query_index=query_index, top_k=5)
        try:
            with io.BytesIO() as buffer:
                fig.savefig(buffer, format="png", dpi=100, bbox_inches="tight")
                buffer.seek(0)
                with Image.open(buffer) as rendered:
                    comparison_images.append(wandb.Image(
                        rendered.convert("RGB"),
                        caption=f"Query {query_index}: ID {table_records[query_index]['identity']}",
                    ))
            if query_index == 0:
                plt.show()  # display one comparison in this evaluation cell too
        finally:
            plt.close(fig)
    
    payload = {"test/evaluation_index": evaluation_index, "test/query_top5_retrievals": retrieval_table}
    if comparison_images:
        payload["test/query_top5_images"] = comparison_images
    else:
        print("No retrieval records were returned; no comparison images to log.")
    run.log(payload)
    print(f"Logged {len(table_records)} query rows and {len(comparison_images)} comparison images to {run.url}")
finally:
    # Finish/sync the scalar record even if table rendering or media logging fails.
    run.finish()


## Inspect query and top-five predictions

The W&B table above contains one row per query. This cell also displays the freshly evaluated results inline. Change `QUERY_INDEX` to browse; rerunning this plot does not repeat evaluation or require an active W&B run.


In [ ]:
import matplotlib.pyplot as plt
from prw_evaluation import plot_retrieval_query

QUERY_INDEX = 0
TOP_K = 5
if table_records:
    fig = plot_retrieval_query(table_records, PRW_ROOT, query_index=QUERY_INDEX, top_k=TOP_K)
    plt.show()
else:
    print("No retrieval records available for visualization.")


## Optional: isolate ReID from detection (no training)
Run after the evaluation cell has loaded the desired checkpoint. This repeats detected-crop evaluation and evaluates GT crops with the **same in-memory model**, official test queries and full gallery. All annotated people, including unknown-ID distractors, remain in the GT gallery. Query source frames are excluded in both modes.

The GT result measures retrieval with annotated boxes; it is not another end-to-end test score or a strict upper bound. Detection precision/recall are computed separately at IoU 0.50 and 0.75. Use this test comparison for diagnosis, and use validation data for choosing subsequent losses/hyperparameters.

Results are saved locally before optional W&B logging. Logging uses a separate `diagnostics/*` namespace in the existing run so the end-to-end test scores remain identifiable.


In [ ]:
import importlib
import json
from pathlib import Path
import pandas as pd
import prw_evaluation
importlib.reload(prw_evaluation)
from prw_evaluation import evaluate_prw, make_wandb_retrieval_table, plot_retrieval_query

if 'model' not in globals() or 'provenance' not in globals():
    raise RuntimeError('First load the desired checkpoint using the evaluation cell above.')

comparison_rows = []
reid_diagnostic_records = {}
for mode in ('detected', 'gt'):
    result = evaluate_prw(
        model, PRW_ROOT, device, gallery_mode=mode,
        score_threshold=.5, iou_threshold=.5,
        batch_size=2, query_batch_size=64, ranking_batch_size=8,
        gallery_crop_batch_size=32, return_records=True,
        table_queries=100, table_top_k=5,
    )
    reid_diagnostic_records[mode] = result.pop('retrieval_records')
    comparison_rows.append(result)

comparison = pd.DataFrame(comparison_rows)
display(comparison)
output = Path('/kaggle/working/evaluation_results') / f'{ckpt_path.parent.name}-epoch-{provenance["test/checkpoint_epoch"]}-reid-diagnostic.json'
output.parent.mkdir(parents=True, exist_ok=True)
output.write_text(json.dumps({'provenance': provenance, 'results': comparison_rows,
                             'records': reid_diagnostic_records}, indent=2))
print('Saved:', output)
for row in comparison_rows:
    print(f"{row['gallery_mode']}: mAP={row['retrieval_mAP']:.4f}, top1={row['retrieval_top1']:.4f}")

# Inline comparisons use the same query in each gallery mode.
import matplotlib.pyplot as plt
for mode in ('detected', 'gt'):
    if reid_diagnostic_records[mode]:
        fig = plot_retrieval_query(reid_diagnostic_records[mode], PRW_ROOT, query_index=0)
        fig.suptitle(mode + ' gallery crops')
        plt.show()
        plt.close(fig)


In [ ]:
# Optional: log the comparison to the same run as the preceding evaluation.
# Run this cell after computing the comparison above. No inference is repeated.
LOG_DIAGNOSTICS = True
if LOG_DIAGNOSTICS:
    diagnostic_run = wandb.init(entity=run.entity, project=run.project,
                               id=run.id, resume='must') if wandb.run is None else wandb.run
    try:
        scalar_payload = {}
        for row in comparison_rows:
            for key, value in row.items():
                if isinstance(value, (int, float)):
                    scalar_payload[f"diagnostics/{row['gallery_mode']}/{key}"] = value
        scalar_payload['diagnostics/checkpoint_path'] = str(ckpt_path)
        scalar_payload['diagnostics/checkpoint_epoch'] = provenance['test/checkpoint_epoch']
        diagnostic_run.summary.update(scalar_payload)
        diagnostic_run.log(scalar_payload)
        scores = wandb.Table(columns=['mode_metric', 'percent'], data=[
            [f"{r['gallery_mode']} {label}", 100*r[key]]
            for r in comparison_rows
            for label, key in [('mAP', 'retrieval_mAP'), ('Top1', 'retrieval_top1')]])
        diagnostic_run.log({'diagnostics/scores': scores,
            'diagnostics/scores_chart': wandb.plot.bar(scores, 'mode_metric', 'percent',
                                                     title='Detected vs GT crop retrieval (%)')})
        for mode, records in reid_diagnostic_records.items():
            diagnostic_run.log({f'diagnostics/{mode}/retrievals':
                                make_wandb_retrieval_table(records, PRW_ROOT)})
    finally:
        diagnostic_run.finish()
